
# Notebook 08 — Preprocessing / Encoding / Scaling

## Objective
Prepare the train, validation, and test splits for machine-learning models while preventing data leakage.

### Rules
- Use the split files created in Notebook 07.
- Fit imputers, encoders, and scalers **only on the training data**.
- Apply the fitted transformations to validation and test data.
- Do not use the test set to fit or select preprocessing parameters.
- Keep the target (`Disruption_Occurred`) separate.
- Keep `Shipment_ID` out of the model predictors.
- Preserve the same transformed feature columns across train/validation/test.
- Save the fitted preprocessing pipeline so the same transformation can later be used by models and inference.


In [1]:

# Cell 2 — Imports

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:

# Cell 3 — Paths

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "processed" / "splits"
MODEL_DIR = PROJECT_ROOT / "models" / "supply_chain"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Split directory:", SPLIT_DIR)
print("Model directory:", MODEL_DIR)


Project root: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Split directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\splits
Model directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain


In [3]:

# Cell 4 — Load the split data

X_train = pd.read_csv(SPLIT_DIR / "X_train.csv")
y_train = pd.read_csv(SPLIT_DIR / "y_train.csv").squeeze("columns")

X_validation = pd.read_csv(SPLIT_DIR / "X_validation.csv")
y_validation = pd.read_csv(SPLIT_DIR / "y_validation.csv").squeeze("columns")

X_test = pd.read_csv(SPLIT_DIR / "X_test.csv")
y_test = pd.read_csv(SPLIT_DIR / "y_test.csv").squeeze("columns")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_validation:", X_validation.shape, "y_validation:", y_validation.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


X_train: (3500, 49) y_train: (3500,)
X_validation: (750, 49) y_validation: (750,)
X_test: (750, 49) y_test: (750,)


In [4]:

# Cell 5 — Basic input validation

TARGET = "Disruption_Occurred"
ID_COLUMN = "Shipment_ID"

assert TARGET not in X_train.columns
assert TARGET not in X_validation.columns
assert TARGET not in X_test.columns

assert ID_COLUMN not in X_train.columns
assert ID_COLUMN not in X_validation.columns
assert ID_COLUMN not in X_test.columns

assert list(X_train.columns) == list(X_validation.columns) == list(X_test.columns)

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

print("Input validation: PASS")


Input validation: PASS


In [5]:

# Cell 6 — Identify numeric and categorical predictors

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

print("\nNumeric columns:")
print(numeric_features)


Numeric features: 43
Categorical features: 6

Categorical columns:
['Date', 'Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Weather_Condition']

Numeric columns:
['Distance_km', 'Weight_MT', 'Fuel_Price_Index', 'Geopolitical_Risk_Score', 'Carrier_Reliability_Score', 'Lead_Time_Days', 'year', 'month', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'distance_per_mt', 'fuel_risk_interaction', 'reliability_risk_inverse', 'weather_risk_flag', 'long_lead_time_flag', 'shipping_year', 'shipping_month', 'shipping_baltic_dry_index', 'shipping_container_rate_usd_40ft', 'shipping_air_cargo_rate_usd_kg', 'shipping_bdi_mom_change_pct', 'shipping_container_yoy_pct', 'shipping_tanker_rate_aframax_usd_day', 'shipping_bulk_carrier_handysize_usd_day', 'shipping_supply_chain_pressure_index', 'shipping_on_time_delivery_pct', 'commodity_price', 'tariff_tariff_rate_pct', 'tariff_estimated_value_usd_bn', 'tarif

In [6]:

# Cell 7 — Inspect missing values BEFORE preprocessing

print("Missing values in X_train:", int(X_train.isna().sum().sum()))
print("Missing values in X_validation:", int(X_validation.isna().sum().sum()))
print("Missing values in X_test:", int(X_test.isna().sum().sum()))

missing_train = X_train.isna().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

if len(missing_train):
    print("\nTraining columns containing missing values:")
    print(missing_train)
else:
    print("\nNo missing values in X_train.")


Missing values in X_train: 28190
Missing values in X_validation: 10986
Missing values in X_test: 8916

Training columns containing missing values:
tariff_is_retaliation                      1855
tariff_is_section_232                      1855
tariff_is_section_301                      1855
tariff_tariff_rate_pct                     1855
tariff_is_ieepa                            1855
tariff_is_biden                            1855
tariff_is_trump_1_0                        1855
tariff_estimated_value_usd_bn              1855
tariff_is_trump_2_0                        1855
shipping_baltic_dry_index                  1045
shipping_month                             1045
shipping_year                              1045
shipping_container_rate_usd_40ft           1045
shipping_on_time_delivery_pct              1045
shipping_bulk_carrier_handysize_usd_day    1045
shipping_supply_chain_pressure_index       1045
shipping_bdi_mom_change_pct                1045
shipping_container_yoy_pct           

In [7]:

# Cell 8 — Define leakage-safe preprocessing

# Numeric:
#   1. Median imputation fitted only on X_train
#   2. Standard scaling fitted only on X_train
#
# Categorical:
#   1. Most-frequent imputation fitted only on X_train
#   2. One-hot encoding fitted only on X_train
#
# handle_unknown="ignore" allows validation/test categories that were not
# present in training to be transformed without causing an error.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Leakage-safe preprocessing pipeline defined.")


Leakage-safe preprocessing pipeline defined.


In [8]:

# Cell 9 — Fit ONLY on training data

preprocessor.fit(X_train)

print("Preprocessor fitted using X_train only: PASS")


Preprocessor fitted using X_train only: PASS


In [9]:

# Cell 10 — Transform train, validation, and test

X_train_processed = preprocessor.transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_validation_processed.shape)
print("Processed test:", X_test_processed.shape)


Processed train: (3500, 590)
Processed validation: (750, 590)
Processed test: (750, 590)


In [10]:

# Cell 11 — Convert transformed arrays to DataFrames

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Number of transformed features:", len(feature_names))
print("Feature names unique:", pd.Index(feature_names).is_unique)


Number of transformed features: 590
Feature names unique: True


In [11]:

# Cell 12 — Verify transformed feature consistency

print("Train columns == validation columns:",
      list(X_train_processed.columns) == list(X_validation_processed.columns))

print("Train columns == test columns:",
      list(X_train_processed.columns) == list(X_test_processed.columns))

assert list(X_train_processed.columns) == list(X_validation_processed.columns)
assert list(X_train_processed.columns) == list(X_test_processed.columns)

print("Transformed feature consistency: PASS")


Train columns == validation columns: True
Train columns == test columns: True
Transformed feature consistency: PASS


In [12]:

# Cell 13 — Verify numeric quality after preprocessing

def numeric_quality(df):
    return {
        "missing_values": int(df.isna().sum().sum()),
        "infinite_values": int(np.isinf(df.to_numpy(dtype=float)).sum()),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1])
    }

print("Train:", numeric_quality(X_train_processed))
print("Validation:", numeric_quality(X_validation_processed))
print("Test:", numeric_quality(X_test_processed))

assert X_train_processed.isna().sum().sum() == 0
assert X_validation_processed.isna().sum().sum() == 0
assert X_test_processed.isna().sum().sum() == 0

assert np.isfinite(X_train_processed.to_numpy(dtype=float)).all()
assert np.isfinite(X_validation_processed.to_numpy(dtype=float)).all()
assert np.isfinite(X_test_processed.to_numpy(dtype=float)).all()

print("Missing/infinite value check: PASS")


Train: {'missing_values': 0, 'infinite_values': 0, 'rows': 3500, 'columns': 590}
Validation: {'missing_values': 0, 'infinite_values': 0, 'rows': 750, 'columns': 590}
Test: {'missing_values': 0, 'infinite_values': 0, 'rows': 750, 'columns': 590}
Missing/infinite value check: PASS


In [13]:

# Cell 14 — Check scaling on training numeric features

# This check is informational. After StandardScaler, non-constant numeric
# training features should generally have mean near 0 and standard deviation
# near 1. Constant features can remain at standard deviation 0.

scaled_numeric = X_train_processed[numeric_features]

numeric_means = scaled_numeric.mean()
numeric_stds = scaled_numeric.std(ddof=0)

print("Maximum absolute training mean:",
      float(np.abs(numeric_means).max()))

print("Training standard deviation range:",
      float(numeric_stds.min()), "to", float(numeric_stds.max()))

print("\nConstant numeric features after scaling:")
print(numeric_stds[numeric_stds == 0].index.tolist())


Maximum absolute training mean: 5.964054646970648e-14
Training standard deviation range: 0.0 to 1.0000000000000002

Constant numeric features after scaling:
['shipping_year', 'tariff_estimated_value_usd_bn', 'tariff_is_trump_1_0', 'tariff_is_section_301']


In [14]:

# Cell 15 — Target validation

print("y_train values:")
print(y_train.value_counts().sort_index())

print("\ny_validation values:")
print(y_validation.value_counts().sort_index())

print("\ny_test values:")
print(y_test.value_counts().sort_index())

assert set(pd.unique(y_train)).issubset({0, 1})
assert set(pd.unique(y_validation)).issubset({0, 1})
assert set(pd.unique(y_test)).issubset({0, 1})

print("\nTarget validation: PASS")


y_train values:
Disruption_Occurred
0    1343
1    2157
Name: count, dtype: int64

y_validation values:
Disruption_Occurred
0    297
1    453
Name: count, dtype: int64

y_test values:
Disruption_Occurred
0    297
1    453
Name: count, dtype: int64

Target validation: PASS


In [15]:

# Cell 16 — Save processed datasets

X_train_processed.to_csv(DATA_DIR / "processed" / "X_train_preprocessed.csv", index=False)
y_train.to_csv(DATA_DIR / "processed" / "y_train_preprocessed.csv", index=False)

X_validation_processed.to_csv(DATA_DIR / "processed" / "X_validation_preprocessed.csv", index=False)
y_validation.to_csv(DATA_DIR / "processed" / "y_validation_preprocessed.csv", index=False)

X_test_processed.to_csv(DATA_DIR / "processed" / "X_test_preprocessed.csv", index=False)
y_test.to_csv(DATA_DIR / "processed" / "y_test_preprocessed.csv", index=False)

print("Processed datasets saved.")


Processed datasets saved.


In [16]:

# Cell 17 — Save the fitted preprocessing pipeline

preprocessor_path = MODEL_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

print("Saved:", preprocessor_path)


Saved: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\preprocessor.joblib


In [17]:

# Cell 18 — Save transformed feature names

feature_names_path = MODEL_DIR / "preprocessed_feature_names.json"

with open(feature_names_path, "w", encoding="utf-8") as f:
    json.dump(feature_names.tolist(), f, indent=2)

print("Saved:", feature_names_path)


Saved: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\preprocessed_feature_names.json


In [18]:

# Cell 19 — Final validation

expected_rows = {
    "train": 3500,
    "validation": 750,
    "test": 750
}

final_checks = {
    "train_rows": len(X_train_processed),
    "validation_rows": len(X_validation_processed),
    "test_rows": len(X_test_processed),
    "train_feature_count": X_train_processed.shape[1],
    "validation_feature_count": X_validation_processed.shape[1],
    "test_feature_count": X_test_processed.shape[1],
    "feature_columns_identical": (
        list(X_train_processed.columns) ==
        list(X_validation_processed.columns) ==
        list(X_test_processed.columns)
    ),
    "train_missing": int(X_train_processed.isna().sum().sum()),
    "validation_missing": int(X_validation_processed.isna().sum().sum()),
    "test_missing": int(X_test_processed.isna().sum().sum()),
    "train_infinite": int(np.isinf(X_train_processed.to_numpy(dtype=float)).sum()),
    "validation_infinite": int(np.isinf(X_validation_processed.to_numpy(dtype=float)).sum()),
    "test_infinite": int(np.isinf(X_test_processed.to_numpy(dtype=float)).sum()),
    "preprocessor_exists": preprocessor_path.exists(),
    "feature_names_exists": feature_names_path.exists(),
    "X_train_output_exists": (DATA_DIR / "processed" / "X_train_preprocessed.csv").exists(),
    "X_validation_output_exists": (DATA_DIR / "processed" / "X_validation_preprocessed.csv").exists(),
    "X_test_output_exists": (DATA_DIR / "processed" / "X_test_preprocessed.csv").exists()
}

print(pd.Series(final_checks))

assert final_checks["train_rows"] == expected_rows["train"]
assert final_checks["validation_rows"] == expected_rows["validation"]
assert final_checks["test_rows"] == expected_rows["test"]

assert final_checks["feature_columns_identical"]
assert final_checks["train_missing"] == 0
assert final_checks["validation_missing"] == 0
assert final_checks["test_missing"] == 0
assert final_checks["train_infinite"] == 0
assert final_checks["validation_infinite"] == 0
assert final_checks["test_infinite"] == 0

assert final_checks["preprocessor_exists"]
assert final_checks["feature_names_exists"]
assert final_checks["X_train_output_exists"]
assert final_checks["X_validation_output_exists"]
assert final_checks["X_test_output_exists"]

print("\nNotebook 08 final validation: PASS")


train_rows                    3500
validation_rows                750
test_rows                      750
train_feature_count            590
validation_feature_count       590
test_feature_count             590
feature_columns_identical     True
train_missing                    0
validation_missing               0
test_missing                     0
train_infinite                   0
validation_infinite              0
test_infinite                    0
preprocessor_exists           True
feature_names_exists          True
X_train_output_exists         True
X_validation_output_exists    True
X_test_output_exists          True
dtype: object

Notebook 08 final validation: PASS



## Expected interpretation

Do not expect a fixed number of transformed features before running the notebook.

The final feature count depends on the actual categorical levels present in the training split. One-hot encoding will expand categorical variables, while numeric variables remain one column each.

### Important
- No model is trained in this notebook.
- No accuracy, F1, ROC-AUC, RMSE, or other model results are produced here.
- Validation and test data are transformed using the training-fitted preprocessing pipeline.
- The test set remains untouched for model selection.
- The saved `preprocessor.joblib` should be reused later for model training/evaluation and deployment.
